In [3]:
!pip install -q boxmot trackeval opencv-python numpy pandas huggingface_hub ultralytics

In [4]:
import os
import time
import torch
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi, snapshot_download
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

# ================= CONFIGURATION =================
DATASET_BASE_DIR = Path("/kaggle/input/datasets/hannusiaknysh/uav-detection-mot-dataset")
WORKSPACE_DIR = Path("/kaggle/working")
TRACKERS_OUT_DIR = WORKSPACE_DIR / "trackers"

BYTETRACK_GRID = {
    "track_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "track_buffer": [15, 30, 45, 60],
    "match_thresh": [0.7, 0.8, 0.9]
}

OCSORT_GRID = {
    "det_thresh": [0.3, 0.4, 0.5, 0.6, 0.7],
    "max_age": [15, 30, 45, 60],
    "iou_threshold": [0.2, 0.3, 0.4, 0.5]
}

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
REPO_ID = user_secrets.get_secret("REPO_ID")
LOCAL_HF_DATA_DIR = WORKSPACE_DIR / "drone_dataset"
MOT_DATASET_DIR = WORKSPACE_DIR /"drone_mot_dataset"
SEQMAPS_DIR = MOT_DATASET_DIR / "seqmaps"
YOLO_DETECTIONS_OUT = WORKSPACE_DIR / "yolo_detections"
YOLO_MODEL_PATH = LOCAL_HF_DATA_DIR / "models/yolo26n_final_model_best.engine"

In [8]:
import time
import numpy as np
import configparser
import sys
import logging
from pathlib import Path

# --- Bulletproof Multi-version BoxMOT Imports ---
try:
    from boxmot.trackers import ByteTrack, OcSort
except ImportError:
    try:
        from boxmot.trackers.bbox.bytetrack.bytetrack import ByteTrack
        try:
            from boxmot.trackers.bbox.ocsort.ocsort import OcSort
        except ImportError:
            from boxmot.trackers.bbox.ocsort.ocsort import OCSort as OcSort
    except ImportError:
        try:
            from boxmot.trackers.bytetrack.bytetrack import ByteTrack
            try:
                from boxmot.trackers.ocsort.ocsort import OcSort
            except ImportError:
                from boxmot.trackers.ocsort.ocsort import OCSort as OcSort
        except ImportError:
            try:
                from boxmot import BYTETracker as ByteTrack
                from boxmot import OCSORT as OcSort
            except ImportError as e:
                raise ImportError(
                    "Failed to import trackers. Please check your BoxMOT installation. "
                    f"Original error: {e}"
                )

# --- SILENCE BOXMOT LOGS (Must run AFTER importing boxmot) ---
logging.getLogger("boxmot").setLevel(logging.WARNING)
try:
    from loguru import logger
    # Remove any custom log handlers that BoxMOT initialized during its import
    logger.remove()
    # Add a clean handler to only print warnings/errors to prevent console flooding
    logger.add(sys.stderr, level="ERROR")
except ImportError:
    pass


def get_configured_tracker(tracker_name: str, config: dict, fps: int):
    """Factory to initialize a tracker with dynamic configurations."""
    if tracker_name.lower() == "bytetrack":
        return ByteTrack(
            track_thresh=config.get("track_thresh", 0.5),
            match_thresh=config.get("match_thresh", 0.8),
            track_buffer=int(config.get("track_buffer", fps))
        )
    elif tracker_name.lower() == "ocsort":
        return OcSort(
            det_thresh=config.get("det_thresh", 0.5),
            max_age=int(config.get("max_age", fps)),
            iou_threshold=config.get("iou_threshold", 0.3)
        )
    else:
        raise ValueError(f"Unknown tracker type: {tracker_name}")

def run_tracking_pipeline(split_name: str, tracker_name: str, tracker_config: dict, 
                          run_dir_name: str, detections_source_dir: Path = None):
    """
    Runs tracking and measures high-precision tracker execution time (latency and FPS).
    Iterates through ALL frames to ensure correct tracker timing updates.
    """
    split_dir = DATASET_BASE_DIR / f"drone-{split_name}"
    if not split_dir.exists():
        print(f"Error: Split directory {split_dir} does not exist.")
        return
        
    sequences = [d for d in split_dir.iterdir() if d.is_dir()]
    
    # Track global performance statistics across all sequences in this run
    global_total_time = 0.0
    global_total_frames = 0
    
    for seq in sequences:
        seq_name = seq.name
        
        # Parse seqinfo.ini to get total sequence length
        config = configparser.ConfigParser()
        config.read(seq / "seqinfo.ini")
        width = int(config['Sequence']['imWidth'])
        height = int(config['Sequence']['imHeight'])
        fps = int(config['Sequence']['frameRate'])
        seq_length = int(config['Sequence']['seqLength']) # Зчитуємо загальну кількість кадрів
        
        tracker = get_configured_tracker(tracker_name, tracker_config, fps)
        
        # Load detections
        if detections_source_dir is None:
            det_file = seq / "gt/gt.txt"
        else:
            det_file = detections_source_dir / f"{seq_name}.txt"
            
        if not det_file.exists():
            continue
            
        # Завантажуємо детекції, обробляємо порожні файли
        if det_file.stat().st_size > 0:
            detections = np.loadtxt(det_file, delimiter=',')
            if detections.ndim == 1:
                detections = np.array([detections])
        else:
            detections = np.empty((0, 9))
            
        results = []
        dummy_frame = np.zeros((height, width, 3), dtype=np.uint8)
        
        seq_tracker_time = 0.0
        seq_total_frames = 0
        
        # Ітеруємося по ВСІХ кадрах відео (від 1 до seq_length включно)
        for frame_id in range(1, seq_length + 1):
            
            # Витягуємо детекції тільки для поточного кадру, якщо вони існують
            if len(detections) > 0:
                frame_dets = detections[detections[:, 0] == frame_id]
            else:
                frame_dets = np.empty((0, 9))
                
            if frame_dets.ndim == 1:
                frame_dets = np.array([frame_dets])
                
            tracker_input = []
            if len(frame_dets) > 0:
                for det in frame_dets:
                    left, top, w, h = det[2:6]
                    conf = det[6] if (detections_source_dir is not None and det[6] > 0) else 1.0
                    cls = 1
                    tracker_input.append([left, top, left + w, top + h, conf, cls])
                
            # Якщо список пустий (на кадрі нічого немає), робимо правильну форму масиву
            tracker_input = np.array(tracker_input) if len(tracker_input) > 0 else np.empty((0, 6))
            
            # Оновлюємо трекер (це відбудеться і для порожнього масиву)
            start_time = time.perf_counter()
            online_targets = tracker.update(tracker_input, dummy_frame)
            end_time = time.perf_counter()
            
            seq_tracker_time += (end_time - start_time)
            seq_total_frames += 1
            
            # Записуємо результати для знайдених треків
            for target in online_targets:
                x1, y1, x2, y2, track_id, conf, cls = target[:7]
                results.append([frame_id, track_id, x1, y1, x2 - x1, y2 - y1, conf, -1, -1, -1])
                
        # Accumulate global statistics
        global_total_time += seq_tracker_time
        global_total_frames += seq_total_frames
        
        # Save results
        out_path = TRACKERS_OUT_DIR / f"drone-{split_name}" / run_dir_name / "data"
        out_path.mkdir(parents=True, exist_ok=True)
        
        res_file = out_path / f"{seq_name}.txt"
        if len(results) > 0:
            np.savetxt(res_file, np.array(results), fmt='%d,%d,%.2f,%.2f,%.2f,%.2f,%.2f,%d,%d,%d')
        else:
            open(res_file, 'w').close()
            
    # Print global performance metrics
    if global_total_frames > 0:
        global_latency_ms = (global_total_time / global_total_frames) * 1000
        global_fps = global_total_frames / global_total_time
        print(f"===> [{tracker_name.upper()}] {run_dir_name.upper()} | Config: {tracker_config} | Avg Latency: {global_latency_ms:.3f} ms/frame | Avg FPS: {global_fps:.1f}")

In [6]:
import sys
import trackeval

def run_trackeval(split_name: str, trackers_to_eval: list):
    """Runs TrackEval silently, saving all results to disk without console printing."""
    eval_config = trackeval.Evaluator.get_default_eval_config()
    
    # Configure evaluator to run in absolute silent mode
    eval_config.update({
        'PLOT_CURVES': False,
        'PRINT_RESULTS': False,       # Completely disable printing evaluation tables to console
        'PRINT_ONLY_COMBINED': True,
        'TIME_PROGRESS': False,       # Mute sequence-by-sequence time progress
        'PRINT_CONFIG': False         # Mute configuration blocks
    })
    
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    
    # TrackEval outputs config
    dataset_config.update({
        'GT_FOLDER': str(DATASET_BASE_DIR),
        'TRACKERS_FOLDER': str(TRACKERS_OUT_DIR),
        'SUB_SPLIT': '',
        'SPLIT_TO_EVAL': split_name,
        'BENCHMARK': 'drone',
        'TRACKERS_TO_EVAL': trackers_to_eval,
        'CLASSES_TO_EVAL': ['pedestrian'],
        'SKIP_SPLIT_FOL': False,
        'PRINT_CONFIG': False
    })
    
    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
    metrics = [
        trackeval.metrics.HOTA(),
        trackeval.metrics.CLEAR(),
        trackeval.metrics.Identity()
    ]
    
    # Run evaluation quietly
    print(f"Evaluating {len(trackers_to_eval)} configurations on split '{split_name}'...")
    evaluator.evaluate([dataset], metrics)
    print("Evaluation complete. Results saved to disk successfully.")

In [9]:
import itertools
import pandas as pd
from pathlib import Path
from IPython.display import display

# Configure Pandas display settings for wide comparative tables
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def parse_trackeval_summary(summary_path: Path):
    """
    Parses TrackEval summary file, normalizes header percentage signs, 
    and returns a dictionary of metrics.
    """
    if not summary_path.exists():
        return None
        
    with open(summary_path, 'r') as f:
        # Read and filter out any empty lines
        lines = [line.strip() for line in f.readlines() if line.strip()]
        
    if len(lines) < 2:
        return None
        
    headers = lines[0].split()
    values = lines[1].split()
    
    # Normalize headers by removing potential '(%)' suffix to prevent key mismatches
    normalized_headers = [h.replace('(%)', '') for h in headers]
    
    metrics = {}
    for h, v in zip(normalized_headers, values):
        try:
            metrics[h] = float(v)
        except ValueError:
            metrics[h] = v
    return metrics

def find_and_display_best_configs():
    # 1. Re-generate identical parameter combinations to match run indices
    bytetrack_combinations = []
    keys_bt, values_bt = zip(*BYTETRACK_GRID.items())
    bytetrack_combinations = [dict(zip(keys_bt, v)) for v in itertools.product(*values_bt)]
    
    ocsort_combinations = []
    keys_oc, values_oc = zip(*OCSORT_GRID.items())
    ocsort_combinations = [dict(zip(keys_oc, v)) for v in itertools.product(*values_oc)]
    
    # 2. Parse and evaluate ByteTrack runs
    best_bt_hota = -1.0
    best_bt_metrics = None
    best_bt_config = None
    
    for idx, config in enumerate(bytetrack_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"bytetrack_run_{idx}" / "pedestrian_summary.txt"
        metrics = parse_trackeval_summary(summary_file)
        if metrics:
            hota = metrics.get("HOTA", 0.0)
            if hota > best_bt_hota:
                best_bt_hota = hota
                best_bt_metrics = metrics
                best_bt_config = config
                
    # 3. Parse and evaluate OC-SORT runs
    best_oc_hota = -1.0
    best_oc_metrics = None
    best_oc_config = None
    
    for idx, config in enumerate(ocsort_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"ocsort_run_{idx}" / "pedestrian_summary.txt"
        metrics = parse_trackeval_summary(summary_file)
        if metrics:
            hota = metrics.get("HOTA", 0.0)
            if hota > best_oc_hota:
                best_oc_hota = hota
                best_oc_metrics = metrics
                best_oc_config = config

    # 4. Construct and display a clean consolidated comparison table with all major metrics
    if best_bt_metrics or best_oc_metrics:
        print("\n" + "="*115)
        print("                 FINAL COMPARATIVE METRICS REPORT (BEST CALIBRATED CONFIGURATIONS)")
        print("="*115)
        
        comparison_data = []
        
        if best_bt_metrics:
            comparison_data.append({
                "Tracker": "ByteTrack",
                "Conf Thresh": best_bt_config.get("track_thresh"),
                "Memory (Frames)": best_bt_config.get("track_buffer"),
                "IoU Thresh": best_bt_config.get("match_thresh"),
                "HOTA": best_bt_metrics.get("HOTA", 0.0),
                "DetA": best_bt_metrics.get("DetA", 0.0),
                "AssA": best_bt_metrics.get("AssA", 0.0),
                "IDF1": best_bt_metrics.get("IDF1", 0.0),
                "MOTA": best_bt_metrics.get("MOTA", 0.0),
                "MOTP": best_bt_metrics.get("MOTP", 0.0),
                "FP": int(best_bt_metrics.get("CLR_FP", 0)),
                "FN": int(best_bt_metrics.get("CLR_FN", 0)),
                "IDSW": int(best_bt_metrics.get("IDSW", 0)),
                "Frag": int(best_bt_metrics.get("Frag", 0))
            })
            
        if best_oc_metrics:
            comparison_data.append({
                "Tracker": "OC-SORT",
                "Conf Thresh": best_oc_config.get("det_thresh"),
                "Memory (Frames)": best_oc_config.get("max_age"),
                "IoU Thresh": best_oc_config.get("iou_threshold"),
                "HOTA": best_oc_metrics.get("HOTA", 0.0),
                "DetA": best_oc_metrics.get("DetA", 0.0),
                "AssA": best_oc_metrics.get("AssA", 0.0),
                "IDF1": best_oc_metrics.get("IDF1", 0.0),
                "MOTA": best_oc_metrics.get("MOTA", 0.0),
                "MOTP": best_oc_metrics.get("MOTP", 0.0),
                "FP": int(best_oc_metrics.get("CLR_FP", 0)),
                "FN": int(best_oc_metrics.get("CLR_FN", 0)),
                "IDSW": int(best_oc_metrics.get("IDSW", 0)),
                "Frag": int(best_oc_metrics.get("Frag", 0))
            })
            
        df = pd.DataFrame(comparison_data)
        
        # Formatter dictionary for percentage string presentation
        float_format = {
            "HOTA": "{:.2f}%", "DetA": "{:.2f}%", "AssA": "{:.2f}%",
            "IDF1": "{:.2f}%", "MOTA": "{:.2f}%", "MOTP": "{:.2f}%"
        }
        
        # Apply style formatting and hide default pandas index column
        styled_df = df.style.format(float_format).hide(axis='index')
        display(styled_df)
        print("="*115 + "\n")
    else:
        print("Error: No evaluation summary files found. Please ensure the Grid Search completed successfully.")

In [12]:
import itertools

# --- 1. Run ByteTrack Grid ---
bytetrack_runs = []
keys, values = zip(*BYTETRACK_GRID.items())
bytetrack_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("Starting ByteTrack Grid Search...")
for idx, config in enumerate(bytetrack_combinations):
    run_name = f"bytetrack_run_{idx}"
    bytetrack_runs.append(run_name)
    # Store config details in the run name directory structure later or parse via metadata
    run_tracking_pipeline(split_name="val", tracker_name="bytetrack", 
                          tracker_config=config, run_dir_name=run_name)

# Run evaluation for all ByteTrack combinations in one go
run_trackeval(split_name="val", trackers_to_eval=bytetrack_runs)


# --- 2. Run OC-SORT Grid ---
ocsort_runs = []
keys, values = zip(*OCSORT_GRID.items())
ocsort_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("\nStarting OC-SORT Grid Search...")
for idx, config in enumerate(ocsort_combinations):
    run_name = f"ocsort_run_{idx}"
    ocsort_runs.append(run_name)
    run_tracking_pipeline(split_name="val", tracker_name="ocsort", 
                          tracker_config=config, run_dir_name=run_name)

# Run evaluation for all OC-SORT combinations in one go
run_trackeval(split_name="val", trackers_to_eval=ocsort_runs)

Starting ByteTrack Grid Search...
===> [BYTETRACK] BYTETRACK_RUN_0 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.7} | Avg Latency: 0.588 ms/frame | Avg FPS: 1699.6
===> [BYTETRACK] BYTETRACK_RUN_1 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.8} | Avg Latency: 0.585 ms/frame | Avg FPS: 1708.5
===> [BYTETRACK] BYTETRACK_RUN_2 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.9} | Avg Latency: 0.587 ms/frame | Avg FPS: 1702.9
===> [BYTETRACK] BYTETRACK_RUN_3 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.7} | Avg Latency: 0.586 ms/frame | Avg FPS: 1706.7
===> [BYTETRACK] BYTETRACK_RUN_4 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.8} | Avg Latency: 0.584 ms/frame | Avg FPS: 1712.8
===> [BYTETRACK] BYTETRACK_RUN_5 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.9} | Avg Latency: 0.579 ms/frame | Avg FPS: 1726.9
===> [BYTETRACK] BYTETRACK_RUN_6 | Config:

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

===> [OCSORT] OCSORT_RUN_79 | Config: {'det_thresh': 0.7, 'max_age': 60, 'iou_threshold': 0.5} | Avg Latency: 0.637 ms/frame | Avg FPS: 1570.7

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          
Evaluating 80 configurations on split 'val'...

Evaluating 80 tracker(s) on 84 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating ocsort_run_0


Evaluating ocsort_run_1


Evaluating ocsort_run_2


Evaluating ocsort_run_3


Evaluating ocsort_run_4


Evaluating ocsort_run_5


Evaluating ocsort_run_6


Evaluating ocsort_run_7


Evaluating ocsort_run_8


Evaluating ocsort_run_9


Evaluating ocsort_run_10


Evaluating ocsort_run_11


Evaluating ocsort_run_12


Evaluating ocsort_run_13


Evaluating ocsort_run_14


Evaluating ocsort

In [13]:
find_and_display_best_configs()


                 FINAL COMPARATIVE METRICS REPORT (BEST CALIBRATED CONFIGURATIONS)


Tracker,Conf Thresh,Memory (Frames),IoU Thresh,HOTA,DetA,AssA,IDF1,MOTA,MOTP,FP,FN,IDSW,Frag
ByteTrack,0.300000,30,0.900000,85.92%,85.25%,86.98%,96.23%,97.80%,88.41%,71,92,11,14
OC-SORT,0.300000,30,0.200000,96.87%,98.44%,95.33%,96.53%,98.32%,100.00%,0,123,10,0


In [14]:
def find_best_configs():
    # Helper to parse TrackEval summary txt
    def parse_hota(summary_path: Path):
        if not summary_path.exists():
            return 0.0, 0.0, 0.0
        with open(summary_path, 'r') as f:
            lines = f.readlines()
        headers = lines[0].strip().split()
        values = lines[1].strip().split()
        metrics = dict(zip(headers, [float(v) for v in values]))
        return metrics.get("HOTA(%)", 0.0), metrics.get("IDF1(%)", 0.0), metrics.get("MOTA(%)", 0.0)

    # 1. Parse ByteTrack runs
    best_bytetrack_hota = -1.0
    best_bytetrack_idx = -1
    best_bytetrack_metrics = {}
    
    for idx, config in enumerate(bytetrack_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"bytetrack_run_{idx}" / "pedestrian_summary.txt"
        hota, idf1, mota = parse_hota(summary_file)
        if hota > best_bytetrack_hota:
            best_bytetrack_hota = hota
            best_bytetrack_idx = idx
            best_bytetrack_metrics = {"HOTA": hota, "IDF1": idf1, "MOTA": mota}

    best_bytetrack_config = bytetrack_combinations[best_bytetrack_idx]
    
    # 2. Parse OC-SORT runs
    best_ocsort_hota = -1.0
    best_ocsort_idx = -1
    best_ocsort_metrics = {}
    
    for idx, config in enumerate(ocsort_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"ocsort_run_{idx}" / "pedestrian_summary.txt"
        hota, idf1, mota = parse_hota(summary_file)
        if hota > best_ocsort_hota:
            best_ocsort_hota = hota
            best_ocsort_idx = idx
            best_ocsort_metrics = {"HOTA": hota, "IDF1": idf1, "MOTA": mota}

    best_ocsort_config = ocsort_combinations[best_ocsort_idx]

    print("\n==================================================")
    print("BEST CALIBRATED CONFIGURATIONS ON 'VAL' SPLIT:")
    print("==================================================")
    print(f"ByteTrack (Run {best_bytetrack_idx}):")
    print(f"  Params: {best_bytetrack_config}")
    print(f"  Metrics: {best_bytetrack_metrics}")
    print("--------------------------------------------------")
    print(f"OC-SORT (Run {best_ocsort_idx}):")
    print(f"  Params: {best_ocsort_config}")
    print(f"  Metrics: {best_ocsort_metrics}")
    print("==================================================")
    
    return best_bytetrack_config, best_ocsort_config

# Run analysis and save optimal parameters to global variables
best_bytetrack_params, best_ocsort_params = find_best_configs()


BEST CALIBRATED CONFIGURATIONS ON 'VAL' SPLIT:
ByteTrack (Run 0):
  Params: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.7}
  Metrics: {'HOTA': 0.0, 'IDF1': 0.0, 'MOTA': 0.0}
--------------------------------------------------
OC-SORT (Run 0):
  Params: {'det_thresh': 0.3, 'max_age': 15, 'iou_threshold': 0.2}
  Metrics: {'HOTA': 0.0, 'IDF1': 0.0, 'MOTA': 0.0}


In [18]:
print("Running final Stage 1 test with best calibrated parameters...")

# Run optimal ByteTrack
run_tracking_pipeline(
    split_name="test", 
    tracker_name="bytetrack", 
    tracker_config=best_bytetrack_params, 
    run_dir_name="bytetrack_final"
)

# Run optimal OC-SORT
run_tracking_pipeline(
    split_name="test", 
    tracker_name="ocsort", 
    tracker_config=best_ocsort_params, 
    run_dir_name="ocsort_final"
)

# Run evaluation on test split
run_trackeval(split_name="test", trackers_to_eval=["bytetrack_final", "ocsort_final"])

Running final Stage 1 test with best calibrated parameters...
===> [BYTETRACK] BYTETRACK_FINAL | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.7} | Avg Latency: 0.568 ms/frame | Avg FPS: 1759.4
===> [OCSORT] OCSORT_FINAL | Config: {'det_thresh': 0.3, 'max_age': 15, 'iou_threshold': 0.2} | Avg Latency: 0.577 ms/frame | Avg FPS: 1731.6

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          
Evaluating 2 configurations on split 'test'...

Evaluating 2 tracker(s) on 84 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating bytetrack_final


Evaluating ocsort_final

Evaluation complete. Results saved to disk successfully.


In [19]:
bt_test_metrics = parse_trackeval_summary(TRACKERS_OUT_DIR / "drone-test" / "bytetrack_final" / "pedestrian_summary.txt")
oc_test_metrics = parse_trackeval_summary(TRACKERS_OUT_DIR / "drone-test" / "ocsort_final" / "pedestrian_summary.txt")

if bt_test_metrics and oc_test_metrics:
    df_test = pd.DataFrame([bt_test_metrics, oc_test_metrics], index=["ByteTrack (Final Test)", "OC-SORT (Final Test)"])
    display(df_test)
else:
    print("Error: Could not read final test evaluation summaries using your parser.")

,HOTA,DetA,AssA,DetRe,DetPr,AssRe,AssPr,LocA,OWTA,HOTA(0),LocA(0),HOTALocA(0),MOTA,MOTP,MODA,CLR_Re,CLR_Pr,MTR,PTR,MLR,CLR_TP,CLR_FN,CLR_FP,IDSW,MT,PT,ML,Frag,sMOTA,IDF1,IDR,IDP,IDTP,IDFN,IDFP,Dets,GT_Dets,IDs,GT_IDs
ByteTrack (Final Test),83.550,84.775,82.639,88.904,89.655,85.285,92.282,89.451,85.707,95.316,87.691,83.584,97.394,87.975,97.842,98.502,99.334,98.824,1.1765,0.0,8352.0,127.0,56.0,38.0,84.0,1.0,0.0,15.0,85.549,93.350,92.959,93.744,7882.0,597.0,526.0,8408.0,8479.0,121.0,85.0
OC-SORT (Final Test),96.812,98.585,95.071,98.585,100.000,95.071,100.000,100.000,96.812,96.812,100.000,96.812,98.455,100.000,98.585,98.585,100.000,100.000,0.0000,0.0,8359.0,120.0,0.0,11.0,85.0,0.0,0.0,0.0,98.455,96.722,96.037,97.416,8143.0,336.0,216.0,8359.0,8479.0,96.0,85.0


In [3]:
from huggingface_hub import snapshot_download

def download_dataset_from_hf():
    print(f"Downloading required folders from Hugging Face...")
    LOCAL_HF_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    # Download only folders containing test/val videos, labels, and the model
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(LOCAL_HF_DATA_DIR),
        allow_patterns=[
            "test/**", 
            "val/**", 
            "labels/**", 
            "models/yolo26n_final_model_best.engine"
        ]
    )
    print("Download from Hugging Face complete!")

download_dataset_from_hf()

Fetching ... files: 0it [00:00, ?it/s]

Download from Hugging Face complete!


In [4]:
import os
import cv2
import zipfile
import numpy as np

def prepare_kaggle_dataset():
    SEQMAPS_DIR.mkdir(parents=True, exist_ok=True)
    splits = ["test", "val"]
    video_extensions = (".mp4", ".mov", ".m4v")

    # Recursive search for all zip files inside 'labels' directory to handle batches
    labels_root = LOCAL_HF_DATA_DIR / "labels"
    zip_files = {z.stem: z for z in labels_root.rglob("*.zip")}
    print(f"Found {len(zip_files)} label archives in recursive search.")

    for split in splits:
        split_video_dir = LOCAL_HF_DATA_DIR / split
        trackeval_split_name = f"drone-{split}"
        gt_dir = MOT_DATASET_DIR / trackeval_split_name
        gt_dir.mkdir(parents=True, exist_ok=True)
        
        videos = [f for f in split_video_dir.iterdir() if f.suffix.lower() in video_extensions]
        matched_seqs = []
        
        for video_path in videos:
            seq_name = video_path.stem
            # Match using the recursive dictionary
            zip_path = zip_files.get(seq_name) or zip_files.get(f"{seq_name}_annotated")
            
            if zip_path:
                matched_seqs.append(seq_name)
                
                # 1. Get video metadata for seqinfo.ini
                cap = cv2.VideoCapture(str(video_path))
                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                fps = cap.get(cv2.CAP_PROP_FPS)
                length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                
                # 2. Create sequence folder
                seq_gt_dir = gt_dir / seq_name / "gt"
                seq_gt_dir.mkdir(parents=True, exist_ok=True)
                
                # 3. Write seqinfo.ini
                seqinfo_content = f"[Sequence]\nname={seq_name}\nimDir=img1\nframeRate={int(fps)}\nseqLength={length}\nimWidth={width}\nimHeight={height}\nimExt=.jpg\n"
                with open(gt_dir / seq_name / "seqinfo.ini", "w") as f:
                    f.write(seqinfo_content)
                    
                # 4. Extract and normalize annotations directly from ZIP
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    gt_file_in_zip = None
                    for name in zip_ref.namelist():
                        if name.endswith("gt.txt"):
                            gt_file_in_zip = name
                            break
                    
                    if gt_file_in_zip:
                        with zip_ref.open(gt_file_in_zip) as f_stream:
                            gt_data = np.loadtxt(f_stream, delimiter=',')
                        
                        # Enforce class ID to 1 for TrackEval
                        if gt_data.ndim == 2:
                            gt_data[:, 7] = 1
                        elif gt_data.ndim == 1:
                            gt_data[7] = 1
                        
                        target_gt_path = seq_gt_dir / "gt.txt"
                        np.savetxt(target_gt_path, gt_data, fmt='%d,%d,%.2f,%.2f,%.2f,%.2f,%d,%d,%.2f')
                    else:
                        print(f"Warning: gt.txt was not found inside {zip_path.name}")
            else:
                print(f"Warning: No matching zip archive found for video {video_path.name}")
                
        # Write seqmap
        with open(SEQMAPS_DIR / f"{trackeval_split_name}.txt", "w") as f:
            f.write("name\n" + "\n".join(matched_seqs))
            
    print("Dataset compilation for TrackEval complete.")

prepare_kaggle_dataset()

Found 591 label archives in recursive search.
Dataset compilation for TrackEval complete.


In [5]:
from ultralytics import YOLO

def run_yolo26_inference_on_kaggle():
    model = YOLO(str(YOLO_MODEL_PATH), task="detect")
    splits = ["val", "test"]
    video_extensions = (".mp4", ".mov", ".m4v")

    for split in splits:
        split_video_dir = LOCAL_HF_DATA_DIR / split
        output_split_dir = YOLO_DETECTIONS_OUT / split
        output_split_dir.mkdir(parents=True, exist_ok=True)

        seqmap_file = SEQMAPS_DIR / f"drone-{split}.txt"
        with open(seqmap_file, "r") as f:
            sequences = f.read().splitlines()[1:]

        print(f"\n--- Running YOLO26 GPU inference on '{split}' split ---")
        for seq_name in sequences:
            video_path = None
            for ext in video_extensions:
                temp_path = split_video_dir / f"{seq_name}{ext}"
                if temp_path.exists():
                    video_path = temp_path
                    break

            if not video_path:
                continue

            print(f"Inference on: {video_path.name}")
            results = model(str(video_path), stream=True, conf=0.01, verbose=False)
            detections = []

            for frame_idx, result in enumerate(results):
                frame_id = frame_idx + 1
                boxes = result.boxes
                if boxes is not None:
                    xyxy = boxes.xyxy.cpu().numpy()
                    confs = boxes.conf.cpu().numpy()
                    clss = boxes.cls.cpu().numpy()

                    for i in range(len(xyxy)):
                        x1, y1, x2, y2 = xyxy[i]
                        conf = confs[i]
                        cls = int(clss[i])

                        left = x1
                        top = y1
                        w = x2 - x1
                        h = y2 - y1

                        detections.append([frame_id, -1, left, top, w, h, conf, cls, -1])

            out_file = output_split_dir / f"{seq_name}.txt"
            if len(detections) > 0:
                np.savetxt(out_file, np.array(detections), fmt='%d,%d,%.2f,%.2f,%.2f,%.2f,%.4f,%d,%d')
            else:
                open(out_file, 'w').close()

run_yolo26_inference_on_kaggle()


--- Running YOLO26 GPU inference on 'val' split ---
Inference on: delta_thermal_field_136.mp4
Loading /kaggle/working/drone_dataset/models/yolo26n_final_model_best.engine for TensorRT inference...
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 39.59s
Prepared 3 packages in 19.60s
Installed 3 packages in 8ms
 + tensorrt-cu12==11.1.0.106
 + tensorrt-cu12-bindings==11.1.0.106
 + tensorrt-cu12-libs==11.1.0.106

requirements: AutoUpdate success ✅ 60.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

[06/22/2026-12:34:36] [TRT] [I] Loaded engine size: 6 MiB
[06/22/2026-12:34:36] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)
Inference on: delta_thermal_sky_229.mp4
Inference on: wing_rgb_sky_335.mp4
Inference on: wing_rgb_sky_336.m4v
Inference on: win

In [ ]:
from huggingface_hub import HfApi

def upload_detections_to_hf():
    api = HfApi()
    print(f"Uploading 'yolo_detections/' to HF repository...")
    api.upload_folder(
        folder_path=str(YOLO_DETECTIONS_OUT),
        path_in_repo="yolo_detections",
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print("Upload complete!")

upload_detections_to_hf()

In [20]:
def download_detections_from_hf():
    print(f"Downloading saved detections from HF repo...")
    YOLO_DETECTIONS_OUT.mkdir(parents=True, exist_ok=True)
    
    # Download directly into the workspace directory to keep paths flat
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(WORKSPACE_DIR),
        
        allow_patterns=["yolo_detections/**"]
    )
    print("Download from Hugging Face complete.")

download_detections_from_hf()

Fetching ... files: 0it [00:00, ?it/s]

Download from Hugging Face complete.


In [21]:
import itertools
from pathlib import Path

# Fix: Explicitly point to the compiled dataset in /kaggle/working/
YOLO_VAL_DETECTIONS = Path("/kaggle/working/yolo_detections/val")

# 1. Run Grid Search for ByteTrack on YOLO detections
bytetrack_runs_model = []
keys, values = zip(*BYTETRACK_GRID.items())
bytetrack_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("Starting ByteTrack Grid Search on YOLO26 detections...")
for idx, config in enumerate(bytetrack_combinations):
    run_name = f"bytetrack_model_run_{idx}"
    bytetrack_runs_model.append(run_name)
    run_tracking_pipeline(split_name="val", tracker_name="bytetrack", 
                          tracker_config=config, run_dir_name=run_name,
                          detections_source_dir=YOLO_VAL_DETECTIONS)

# Evaluate using YOUR original silent TrackEval
run_trackeval(split_name="val", trackers_to_eval=bytetrack_runs_model)


# 2. Run Grid Search for OC-SORT on YOLO detections
ocsort_runs_model = []
keys, values = zip(*OCSORT_GRID.items())
ocsort_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print("\nStarting OC-SORT Grid Search on YOLO26 detections...")
for idx, config in enumerate(ocsort_combinations):
    run_name = f"ocsort_model_run_{idx}"
    ocsort_runs_model.append(run_name)
    run_tracking_pipeline(split_name="val", tracker_name="ocsort", 
                          tracker_config=config, run_dir_name=run_name,
                          detections_source_dir=YOLO_VAL_DETECTIONS)

run_trackeval(split_name="val", trackers_to_eval=ocsort_runs_model)

print("\nGrid Search on model detections complete. All results saved to disk.")

Starting ByteTrack Grid Search on YOLO26 detections...
===> [BYTETRACK] BYTETRACK_MODEL_RUN_0 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.7} | Avg Latency: 0.608 ms/frame | Avg FPS: 1644.6
===> [BYTETRACK] BYTETRACK_MODEL_RUN_1 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.8} | Avg Latency: 0.609 ms/frame | Avg FPS: 1642.3
===> [BYTETRACK] BYTETRACK_MODEL_RUN_2 | Config: {'track_thresh': 0.3, 'track_buffer': 15, 'match_thresh': 0.9} | Avg Latency: 0.601 ms/frame | Avg FPS: 1664.5
===> [BYTETRACK] BYTETRACK_MODEL_RUN_3 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.7} | Avg Latency: 0.621 ms/frame | Avg FPS: 1610.0
===> [BYTETRACK] BYTETRACK_MODEL_RUN_4 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.8} | Avg Latency: 0.616 ms/frame | Avg FPS: 1623.2
===> [BYTETRACK] BYTETRACK_MODEL_RUN_5 | Config: {'track_thresh': 0.3, 'track_buffer': 30, 'match_thresh': 0.9} | Avg Latency: 0.607 ms/frame | A

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

WARNING  Max age > max observations, increasing size of max observations...

===> [OCSORT] OCSORT_MODEL_RUN_79 | Config: {'det_thresh': 0.7, 'max_age': 60, 'iou_threshold': 0.5} | Avg Latency: 0.695 ms/frame | Avg FPS: 1438.8

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          
Evaluating 80 configurations on split 'val'...

Evaluating 80 tracker(s) on 84 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating ocsort_model_run_0


Evaluating ocsort_model_run_1


Evaluating ocsort_model_run_2


Evaluating ocsort_model_run_3


Evaluating ocsort_model_run_4


Evaluating ocsort_model_run_5


Evaluating ocsort_model_run_6


Evaluating ocsort_model_run_7


Evaluating ocsort_model_run_8


Evaluating ocsort_model_run_9


Evaluating ocsort_model_run_10


Evaluating ocsort_model_run_11


Evaluating ocsort_mo

In [22]:
# We redefine this short analysis block locally to scan the model runs
def find_and_display_best_configs_for_yolo():
    # Parse and evaluate ByteTrack model runs
    best_bt_hota = -1.0
    best_bt_metrics = None
    best_bt_config = None
    
    for idx, config in enumerate(bytetrack_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"bytetrack_model_run_{idx}" / "pedestrian_summary.txt"
        metrics = parse_trackeval_summary(summary_file)
        if metrics:
            hota = metrics.get("HOTA", 0.0)
            if hota > best_bt_hota:
                best_bt_hota = hota
                best_bt_metrics = metrics
                best_bt_config = config
                
    # Parse and evaluate OC-SORT model runs
    best_oc_hota = -1.0
    best_oc_metrics = None
    best_oc_config = None
    
    for idx, config in enumerate(ocsort_combinations):
        summary_file = TRACKERS_OUT_DIR / "drone-val" / f"ocsort_model_run_{idx}" / "pedestrian_summary.txt"
        metrics = parse_trackeval_summary(summary_file)
        if metrics:
            hota = metrics.get("HOTA", 0.0)
            if hota > best_oc_hota:
                best_oc_hota = hota
                best_oc_metrics = metrics
                best_oc_config = config

    # Construct and display your clean comparative table
    if best_bt_metrics or best_oc_metrics:
        print("\n" + "="*115)
        print("                 YOLO26 DETECTIONS COMPARATIVE METRICS REPORT (BEST CONFIGURATIONS)")
        print("="*115)
        
        comparison_data = []
        if best_bt_metrics:
            comparison_data.append({
                "Tracker": "ByteTrack",
                "Conf Thresh": best_bt_config.get("track_thresh"),
                "Memory (Frames)": best_bt_config.get("track_buffer"),
                "IoU Thresh": best_bt_config.get("match_thresh"),
                "HOTA": best_bt_metrics.get("HOTA", 0.0),
                "DetA": best_bt_metrics.get("DetA", 0.0),
                "AssA": best_bt_metrics.get("AssA", 0.0),
                "IDF1": best_bt_metrics.get("IDF1", 0.0),
                "MOTA": best_bt_metrics.get("MOTA", 0.0),
                "MOTP": best_bt_metrics.get("MOTP", 0.0),
                "FP": int(best_bt_metrics.get("CLR_FP", 0)),
                "FN": int(best_bt_metrics.get("CLR_FN", 0)),
                "IDSW": int(best_bt_metrics.get("IDSW", 0)),
                "Frag": int(best_bt_metrics.get("Frag", 0))
            })
            
        if best_oc_metrics:
            comparison_data.append({
                "Tracker": "OC-SORT",
                "Conf Thresh": best_oc_config.get("det_thresh"),
                "Memory (Frames)": best_oc_config.get("max_age"),
                "IoU Thresh": best_oc_config.get("iou_threshold"),
                "HOTA": best_oc_metrics.get("HOTA", 0.0),
                "DetA": best_oc_metrics.get("DetA", 0.0),
                "AssA": best_oc_metrics.get("AssA", 0.0),
                "IDF1": best_oc_metrics.get("IDF1", 0.0),
                "MOTA": best_oc_metrics.get("MOTA", 0.0),
                "MOTP": best_oc_metrics.get("MOTP", 0.0),
                "FP": int(best_oc_metrics.get("CLR_FP", 0)),
                "FN": int(best_oc_metrics.get("CLR_FN", 0)),
                "IDSW": int(best_oc_metrics.get("IDSW", 0)),
                "Frag": int(best_oc_metrics.get("Frag", 0))
            })
            
        df = pd.DataFrame(comparison_data)
        float_format = {
            "HOTA": "{:.2f}%", "DetA": "{:.2f}%", "AssA": "{:.2f}%",
            "IDF1": "{:.2f}%", "MOTA": "{:.2f}%", "MOTP": "{:.2f}%"
        }
        styled_df = df.style.format(float_format).hide(axis='index')
        display(styled_df)
        print("="*115 + "\n")
    else:
        print("Error: No evaluation summary files found for model runs.")

find_and_display_best_configs_for_yolo()


                 YOLO26 DETECTIONS COMPARATIVE METRICS REPORT (BEST CONFIGURATIONS)


Tracker,Conf Thresh,Memory (Frames),IoU Thresh,HOTA,DetA,AssA,IDF1,MOTA,MOTP,FP,FN,IDSW,Frag
ByteTrack,0.600000,15,0.900000,69.26%,66.49%,72.68%,84.14%,77.29%,83.86%,628,1134,34,39
OC-SORT,0.400000,60,0.200000,61.38%,65.77%,57.43%,69.11%,69.99%,89.92%,397,1861,115,18
